# Network Intrusion Detection on NSL-KDD

**Dataset:** NSL-KDD — improved version of KDD Cup 1999 intrusion detection data.  
**Task:** Classify network connections as normal or as one of four attack types: DoS, Probe, R2L, U2R.  
**Approach:** Compare three classifiers (Logistic Regression, Random Forest, Gradient Boosting) on binary and multiclass targets. Train on KDDTrain+, evaluate on KDDTest+ — a harder split intended to test generalization to novel attacks.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, roc_auc_score,
)
from sklearn.model_selection import cross_val_score
from src.data import load_data, _load_raw, _label_to_category
%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted")


## 1. Exploratory Data Analysis

In [ ]:
train_raw = _load_raw("KDDTrain+.txt")
test_raw  = _load_raw("KDDTest+.txt")
print("Train:", train_raw.shape, "  Test:", test_raw.shape)
train_raw.head(3)


In [ ]:
# Top attack types in training set
label_counts = train_raw["label"].value_counts()
print("Unique labels:", len(label_counts))
label_counts.head(15)


In [ ]:
train_raw["category"] = train_raw["label"].apply(_label_to_category)
test_raw["category"]  = test_raw["label"].apply(_label_to_category)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (df, split) in zip(axes, [(train_raw, "Train"), (test_raw, "Test")]):
    counts = df["category"].value_counts()
    ax.bar(counts.index, counts.values, color="steelblue")
    ax.set_title(f"Category distribution — {split}")
    ax.set_ylabel("Count")
    for i, (lbl, val) in enumerate(counts.items()):
        ax.text(i, val + 100, str(val), ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("../figures/eda_category_dist.png", dpi=120)
plt.show()
print("U2R and R2L are rare in training but more prevalent in test — a key challenge.")


In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
train_raw["protocol_type"].value_counts().plot(kind="bar", ax=ax, color="steelblue", rot=0)
ax.set_title("Protocol type — train")
plt.tight_layout()
plt.savefig("../figures/eda_protocol.png", dpi=120)
plt.show()


In [ ]:
feats = ["duration", "src_bytes", "dst_bytes", "count", "same_srv_rate"]
fig, axes = plt.subplots(1, len(feats), figsize=(16, 3))
for ax, feat in zip(axes, feats):
    d = train_raw[feat].clip(upper=train_raw[feat].quantile(0.99))
    ax.hist(d, bins=40, color="steelblue", edgecolor="none")
    ax.set_title(feat, fontsize=9)
plt.tight_layout()
plt.savefig("../figures/eda_numeric.png", dpi=120)
plt.show()
print("Most numeric features have heavy right skew — typical for network traffic.")


## 2. Preprocessing

1. Drop `difficulty_level` (metadata, not a feature).  
2. One-hot encode `protocol_type`, `service`, `flag`.  
3. Align train/test columns — test may be missing service categories that exist only in train.  
4. StandardScaler on all numeric (non-dummy) columns.


In [ ]:
X_train, X_test, y_bin_tr, y_bin_te, y_multi_tr, y_multi_te, feat_names = load_data(download=False)
print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")
print(f"Total features after one-hot encoding: {len(feat_names)}")
print()
print("Binary dist (test):")
print(pd.Series(y_bin_te).value_counts().rename({0: "normal", 1: "attack"}))
print()
print("Multiclass dist (test):")
print(pd.Series(y_multi_te).value_counts())


## 3. Model Training and Evaluation

In [ ]:
labels_order = ["normal", "dos", "probe", "r2l", "u2r"]

def train_eval_binary(name, clf):
    clf.fit(X_train, y_bin_tr)
    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else None
    acc  = accuracy_score(y_bin_te, y_pred)
    f1m  = f1_score(y_bin_te, y_pred, average="macro")
    roc  = roc_auc_score(y_bin_te, y_prob) if y_prob is not None else None
    return {"model": name, "acc": acc, "f1_macro": f1m, "roc_auc": roc, "clf": clf}

def train_eval_multi(name, clf):
    clf.fit(X_train, y_multi_tr)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_multi_te, y_pred)
    f1m = f1_score(y_multi_te, y_pred, average="macro", labels=labels_order, zero_division=0)
    per = f1_score(y_multi_te, y_pred, average=None, labels=labels_order, zero_division=0)
    return {"model": name, "acc": acc, "f1_macro": f1m, "per": dict(zip(labels_order, per)), "clf": clf, "pred": y_pred}

bin_models = [
    ("LogisticRegression", LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs", n_jobs=-1)),
    ("RandomForest",       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ("GradientBoosting",   GradientBoostingClassifier(n_estimators=100, random_state=42)),
]
multi_models = [
    ("LogisticRegression", LogisticRegression(max_iter=1000, C=1.0, solver="lbfgs", n_jobs=-1)),
    ("RandomForest",       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ("GradientBoosting",   GradientBoostingClassifier(n_estimators=100, random_state=42)),
]

bin_res   = [train_eval_binary(n, m) for n, m in bin_models]
multi_res = [train_eval_multi(n, m) for n, m in multi_models]

print("Binary results:")
for r in bin_res:
    roc = f"{r['roc_auc']:.4f}" if r['roc_auc'] else "N/A"
    print(f"  {r['model']:<22}  acc={r['acc']:.4f}  f1_macro={r['f1_macro']:.4f}  roc_auc={roc}")
print()
print("Multiclass results:")
for r in multi_res:
    print(f"  {r['model']:<22}  acc={r['acc']:.4f}  f1_macro={r['f1_macro']:.4f}")
    for lbl, v in r['per'].items():
        print(f"    {lbl:<10} {v:.4f}")
    print()


In [ ]:
# Confusion matrix — RandomForest multiclass (best F1 on DoS/Probe)
rf_multi_res = next(r for r in multi_res if r["model"] == "RandomForest")
cm = confusion_matrix(y_multi_te, rf_multi_res["pred"], labels=labels_order)
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels_order, yticklabels=labels_order, ax=ax)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
ax.set_title("RandomForest Multiclass CM (KDDTest+)")
plt.tight_layout()
plt.savefig("../figures/cm_rf_notebook.png", dpi=120)
plt.show()


## 4. Discussion

**What works:** DoS and Probe attacks are classified reliably across all models — both classes are well-represented in training (45 927 and 11 656 samples) and the patterns transfer to the test set.

**Class imbalance:** R2L has ~995 train samples; U2R has only 52. The models see too few examples to learn robust decision boundaries. KDDTest+ has proportionally more of both classes (2 754 R2L, 200 U2R), which amplifies the gap. This is not a bug — it is a well-documented limitation of the dataset.

**Why test accuracy < CV accuracy:** KDDTest+ was deliberately constructed with novel attack variants not present in training. Train cross-validation accuracy (~97–99%) reflects in-distribution performance. Test accuracy (~75–80%) reflects generalization under distribution shift. The gap is real and expected — any project claiming 99%+ on KDDTest+ should be questioned.

**What would improve results:**
- Class-weighted loss (`class_weight="balanced"`) or SMOTE oversampling for R2L/U2R.  
- Feature selection (e.g., dropping near-zero-variance columns like `num_outbound_cmds`).  
- XGBoost or LightGBM — faster and often better on tabular data with distribution shift.  
- A two-stage classifier: first binary normal/attack, then attack-type classification only on flagged connections.
